In [3]:
!pip install jellyfish

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 2.1 MB/s eta 0:00:0000:0100:01


In [4]:
import re
import unicodedata
from collections import defaultdict

import pandas as pd
import numpy as np
import jellyfish
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [5]:
import re
import gc
import unicodedata
from collections import defaultdict

import pandas as pd
import numpy as np
import jellyfish
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

try:
    from tqdm.notebook import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

In [6]:
import os

DATA_ROOT = DATA_ROOT = r"C:\Users\Suyash\Downloads\amazon-ml\student_resource\dataset"  
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR = os.path.join(DATA_ROOT, "test")
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [7]:
import os

p = "/kaggle/input/datasets/suyashma/amazon-ml"
print(os.listdir(p))

['student_resource']


In [8]:
import os

DATA_ROOT = "/kaggle/input/datasets/suyashma/amazon-ml/student_resource/dataset"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR = os.path.join(DATA_ROOT, "test")
OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

check_path = os.path.join(TRAIN_DIR, "train_source1.tsv")
print(f"Looking for: {check_path}")
print(f"Found: {os.path.exists(check_path)}")

Looking for: /kaggle/input/datasets/suyashma/amazon-ml/student_resource/dataset/train/train_source1.tsv
Found: True


In [9]:
"""
Text normalization for business entity resolution.

Handles:
- Unicode NFKC normalization
- Legal-suffix expansion/canonicalization (Corp/Corporation, Pvt/Private, Ltd/Limited, ...)
- Address abbreviation expansion (Rd->Road, St->Street, ...)
- PIN/ZIP code, city, state extraction from free-text addresses (country-aware, with a
  generic fallback so unseen countries like France at test time don't break the pipeline)
"""
import re
import unicodedata


LEGAL_SUFFIX_MAP = {
    r"\bpvt\.?\b": "private",
    r"\bprivate\b": "private",
    r"\bltd\.?\b": "limited",
    r"\blimited\b": "limited",
    r"\bcorp\.?\b": "corporation",
    r"\bcorporation\b": "corporation",
    r"\binc\.?\b": "incorporated",
    r"\bincorporated\b": "incorporated",
    r"\bllc\.?\b": "llc",
    r"\bl\.l\.c\.?\b": "llc",
    r"\bllp\.?\b": "llp",
    r"\bl\.l\.p\.?\b": "llp",
    r"\bco\.?\b": "company",
    r"\bcompany\b": "company",
    r"\bplc\.?\b": "plc",
    r"\bpvt\.?\s*ltd\.?\b": "private limited",
}


LEGAL_SUFFIX_STRIP = [
    "private limited", "pvt ltd", "pvt. ltd.", "limited", "ltd", "ltd.",
    "corporation", "corp", "corp.", "incorporated", "inc", "inc.",
    "llc", "l.l.c", "llp", "l.l.p", "company", "co", "co.", "plc",
    "private", "pvt", "pvt.",
]


ADDRESS_ABBR_MAP = {
    r"\brd\.?\b": "road",
    r"\bst\.?\b": "street",
    r"\bave\.?\b": "avenue",
    r"\bblvd\.?\b": "boulevard",
    r"\bdr\.?\b": "drive",
    r"\bln\.?\b": "lane",
    r"\bapt\.?\b": "apartment",
    r"\bbldg\.?\b": "building",
    r"\bfl\.?\b": "floor",
    r"\bflr\.?\b": "floor",
    r"\bnr\.?\b": "near",
    r"\bopp\.?\b": "opposite",
    r"\bsec\.?\b": "sector",
    r"\bsoc\.?\b": "society",
    r"\bcolny\b": "colony",
    r"\bhwy\.?\b": "highway",
    r"\bmkt\.?\b": "market",
    r"\bind\.?\s*est\.?\b": "industrial estate",
    r"\bp\.?o\.?\b": "post office",
    r"\bdist\.?\b": "district",
    r"\bstn\.?\b": "station",
    r"\bjn\.?\b": "junction",
    r"\bcrossg\b": "crossing",
    r"\bappts?\.?\b": "apartments",
    r"\bste\.?\b": "suite",
}


IN_PIN_RE = re.compile(r"\b(\d{6})\b")
US_ZIP_RE = re.compile(r"\b(\d{5})(?:-\d{4})?\b")
FR_POSTAL_RE = re.compile(r"\b(\d{5})\b")
GENERIC_POSTAL_RE = re.compile(r"\b(\d{4,6})\b")

US_STATE_ABBR = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "ID", "IL", "IN",
    "IA", "KS", "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV",
    "NH", "NJ", "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN",
    "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY", "DC",
}

IN_STATES = {
    "andhra pradesh", "arunachal pradesh", "assam", "bihar", "chhattisgarh", "goa",
    "gujarat", "haryana", "himachal pradesh", "jharkhand", "karnataka", "kerala",
    "madhya pradesh", "maharashtra", "manipur", "meghalaya", "mizoram", "nagaland",
    "odisha", "punjab", "rajasthan", "sikkim", "tamil nadu", "telangana", "tripura",
    "uttar pradesh", "uttarakhand", "west bengal", "delhi", "puducherry", "chandigarh",
    "jammu and kashmir", "ladakh",
}


def nfkc_normalize(text: str) -> str:
    """Unicode NFKC normalization + basic cleanup."""
    if text is None:
        return ""
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    return text


def basic_clean(text: str) -> str:
    """Lowercase, normalize punctuation, collapse whitespace."""
    text = nfkc_normalize(text)
    text = text.lower()
    text = text.replace("&", " and ")
    # Drop most punctuation except digits/letters/space, keep hyphens as spaces
    text = re.sub(r"[.,;:'\"/\\()\[\]{}]", " ", text)
    text = re.sub(r"-", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _apply_map(text: str, mapping: dict) -> str:
    for pattern, repl in mapping.items():
        text = re.sub(pattern, repl, text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_business_name(raw_name: str, strip_suffix: bool = False) -> str:
   
    text = basic_clean(raw_name)
    if strip_suffix:
        # Try longest suffixes first
        for suf in sorted(LEGAL_SUFFIX_STRIP, key=len, reverse=True):
            pattern = r"\b" + re.escape(suf).replace(r"\.", r"\.?") + r"\b\.?\s*$"
            text = re.sub(pattern, "", text).strip()
        text = re.sub(r"\s+", " ", text).strip()
        return text
    else:
        return _apply_map(text, LEGAL_SUFFIX_MAP)


def normalize_address(raw_address: str) -> str:
   
    text = basic_clean(raw_address)
    text = _apply_map(text, ADDRESS_ABBR_MAP)
    return text


def extract_address_components(raw_address: str, country: str) -> dict:
    
    text = nfkc_normalize(raw_address)
    country_norm = (country or "").strip().lower()

    postal_code = None
    state = None

    if country_norm in ("india", "in"):
        m = IN_PIN_RE.search(text)
        if m:
            postal_code = m.group(1)
        lower_text = text.lower()
        for st in IN_STATES:
            if st in lower_text:
                state = st
                break
    elif country_norm in ("united states", "usa", "us"):
        m = US_ZIP_RE.search(text)
        if m:
            postal_code = m.group(1)
        tokens = re.findall(r"\b[A-Za-z]{2}\b", text.upper())
        for tok in tokens:
            if tok in US_STATE_ABBR:
                state = tok
                break
    elif country_norm in ("france", "fr"):
        m = FR_POSTAL_RE.search(text)
        if m:
            postal_code = m.group(1)
        
    else:
        
        m = GENERIC_POSTAL_RE.search(text)
        if m:
            postal_code = m.group(1)

    return {"postal_code": postal_code, "state": state}


def normalize_country(raw_country: str) -> str:
    """Light canonicalization only -- never filter/hard-code to a fixed set."""
    text = basic_clean(raw_country)
    aliases = {
        "usa": "united states", "us": "united states", "u.s.a": "united states",
        "u.s": "united states", "america": "united states",
        "in": "india", "bharat": "india",
        "fr": "france",
    }
    return aliases.get(text, text)


def name_tokens_sorted_key(normalized_core_name: str, n_tokens: int = 3) -> str:
    """
    Sorted-token blocking key: sort the first n_tokens (after removing very short
    stopword-like tokens) alphabetically and join. Robust to word-order transpositions.
    """
    tokens = [t for t in normalized_core_name.split() if len(t) > 1]
    tokens = sorted(tokens)[:n_tokens]
    return " ".join(tokens)


In [10]:
import gc
from collections import defaultdict

try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

import jellyfish
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors



UNKNOWN_COUNTRY_BUCKET = "__unknown__"


def compute_country_labels(df, country_col="country"):
    """Cheap pass: just a Series of normalized country labels, no full-dataframe copy."""
    labels = df[country_col].apply(normalize_country)
    labels = labels.replace("", UNKNOWN_COUNTRY_BUCKET)
    return labels




def build_normalized_fields(df, name_col="business_name", addr_col="business_address",
                             country_col="country"):
    """Attach normalized columns to a dataframe copy. Does not mutate input."""
    df = df.copy()
    df["_country_norm"] = df[country_col].apply(normalize_country)
    df["_country_norm"] = df["_country_norm"].replace("", UNKNOWN_COUNTRY_BUCKET)
    df["_name_core"] = df[name_col].apply(lambda x: normalize_business_name(x, strip_suffix=True))
    df["_name_canon"] = df[name_col].apply(lambda x: normalize_business_name(x, strip_suffix=False))
    df["_addr_norm"] = df[addr_col].apply(normalize_address)
    df["_sorted_key"] = df["_name_core"].apply(name_tokens_sorted_key)

    def _phonetic(name_core):
        tokens = [t for t in name_core.split() if len(t) > 1]
        codes = []
        for t in tokens[:3]:
            try:
                codes.append(jellyfish.metaphone(t))
            except Exception:
                codes.append(t)
        return " ".join(codes)

    df["_phonetic_key"] = df["_name_core"].apply(_phonetic)
    df["_text_for_tfidf"] = (df["_name_canon"] + " " + df["_addr_norm"]).str.strip()
    return df


def normalize_text_fields(df, name_col="business_name", addr_col="business_address"):
   
    return build_normalized_fields(df, name_col=name_col, addr_col=addr_col)



def _group_index_by_key(df, key_col):
    groups = defaultdict(list)
    if df is None or len(df) == 0:
        return groups
    for idx, key in zip(df.index, df[key_col]):
        if key:  # skip empty keys -- they carry no blocking signal
            groups[key].append(idx)
    return groups


def exact_key_candidates(s1_df, other_df, key_col):
    """Return dict: s1_entity_id -> set(other_entity_id) sharing the same key."""
    other_groups = _group_index_by_key(other_df, key_col)
    result = defaultdict(set)
    if s1_df is None or len(s1_df) == 0:
        return result
    for idx, key in zip(s1_df.index, s1_df[key_col]):
        if not key:
            continue
        s1_id = s1_df.at[idx, "entity_id"]
        for other_idx in other_groups.get(key, []):
            result[s1_id].add(other_df.at[other_idx, "entity_id"])
    return result


def tfidf_nn_candidates(
    s1_df,
    other_df,
    top_k=10,
    ngram_range=(2, 4),
    max_features=10000,
    batch_size=500
):
    result = defaultdict(set)

    if (
        s1_df is None or
        other_df is None or
        len(s1_df) == 0 or
        len(other_df) == 0
    ):
        return result

    print(f"      Building TF-IDF index for {len(other_df):,} records...")

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=ngram_range,
        max_features=max_features,
        min_df=2,
        dtype="float32"
    )


    other_vecs = vectorizer.fit_transform(
        other_df["_text_for_tfidf"].fillna("").astype(str)
    )

    print(
        f"      TF-IDF matrix: "
        f"{other_vecs.shape[0]:,} x {other_vecs.shape[1]:,}"
    )

    k = min(top_k, len(other_df))

    nn = NearestNeighbors(
        n_neighbors=k,
        metric="cosine",
        algorithm="brute",
        n_jobs=-1
    )

    nn.fit(other_vecs)

    other_ids = other_df["entity_id"].to_numpy()
    s1_ids = s1_df["entity_id"].to_numpy()

 
    for start in range(0, len(s1_df), batch_size):

        end = min(start + batch_size, len(s1_df))

        batch_text = (
            s1_df["_text_for_tfidf"]
            .iloc[start:end]
            .fillna("")
            .astype(str)
        )

        batch_vecs = vectorizer.transform(batch_text)

        distances, indices = nn.kneighbors(batch_vecs)

        for row_i, s1_id in enumerate(s1_ids[start:end]):

            result[s1_id].update(
                other_ids[indices[row_i]]
            )

        del batch_text
        del batch_vecs
        del distances
        del indices

        if (
            start == 0 or
            end == len(s1_df) or
            (start // batch_size) % 20 == 0
        ):
            print(
                f"      TF-IDF query: "
                f"{end:,}/{len(s1_df):,}"
            )

        gc.collect()

    del other_vecs
    del vectorizer
    del nn

    gc.collect()

    return result

def merge_candidate_dicts(*dicts):
    merged = defaultdict(set)
    for d in dicts:
        for k, v in d.items():
            merged[k] |= v
    return merged


def generate_candidates_for_partition(
    s1_part,
    s2_part,
    s3_part,
    top_k=10,
    tfidf_batch_size=500
):


    print("    S2: phonetic blocking...")

    s2_phonetic = exact_key_candidates(
        s1_part,
        s2_part,
        "_phonetic_key"
    )

    print("    S2: sorted-token blocking...")

    s2_sorted = exact_key_candidates(
        s1_part,
        s2_part,
        "_sorted_key"
    )

    merged_s2 = merge_candidate_dicts(
        s2_phonetic,
        s2_sorted
    )


    del s2_phonetic
    del s2_sorted

    gc.collect()



    print("    S2: TF-IDF blocking...")

    s2_tfidf = tfidf_nn_candidates(
        s1_part,
        s2_part,
        top_k=top_k,
        batch_size=tfidf_batch_size
    )

 
    merged_s2 = merge_candidate_dicts(
        merged_s2,
        s2_tfidf
    )

 
    del s2_tfidf

    gc.collect()



    print("    S3: phonetic blocking...")

    s3_phonetic = exact_key_candidates(
        s1_part,
        s3_part,
        "_phonetic_key"
    )

    print("    S3: sorted-token blocking...")

    s3_sorted = exact_key_candidates(
        s1_part,
        s3_part,
        "_sorted_key"
    )

    merged_s3 = merge_candidate_dicts(
        s3_phonetic,
        s3_sorted
    )

    del s3_phonetic
    del s3_sorted

    gc.collect()

 

    print("    S3: TF-IDF blocking...")

    s3_tfidf = tfidf_nn_candidates(
        s1_part,
        s3_part,
        top_k=top_k,
        batch_size=tfidf_batch_size
    )

    merged_s3 = merge_candidate_dicts(
        merged_s3,
        s3_tfidf
    )

    del s3_tfidf

    gc.collect()


    print("    Merging S2 + S3 candidates...")

    final_candidates = merge_candidate_dicts(
        merged_s2,
        merged_s3
    )

    del merged_s2
    del merged_s3

    gc.collect()

    return final_candidates


def generate_all_candidates(s1_df, s2_df, s3_df, top_k=10):
    print("Labeling countries (lightweight, full dataset)...")
    s1_labels = compute_country_labels(s1_df)
    s2_labels = compute_country_labels(s2_df)
    s3_labels = compute_country_labels(s3_df)

    all_countries = sorted(set(s1_labels) | set(s2_labels) | set(s3_labels))
    total = len(all_countries)
    print(f"Found {total} country partitions.\n")

    candidates = defaultdict(set)

    country_iter = tqdm(all_countries, desc="Countries", unit="country") if HAS_TQDM else all_countries

    for country in country_iter:
        s1_slice = s1_df[s1_labels == country]
        if len(s1_slice) == 0:
            continue
        s2_slice = s2_df[s2_labels == country]
        s3_slice = s3_df[s3_labels == country]

        if HAS_TQDM:
            country_iter.set_postfix_str(f"{country} (S1:{len(s1_slice)} S2:{len(s2_slice)} S3:{len(s3_slice)})")
        else:
            print(f"{country} (S1:{len(s1_slice)} S2:{len(s2_slice)} S3:{len(s3_slice)})")

        s1_part = normalize_text_fields(s1_slice)
        s2_part = normalize_text_fields(s2_slice)
        s3_part = normalize_text_fields(s3_slice)

        part_candidates = generate_candidates_for_partition(s1_part, s2_part, s3_part, top_k=top_k)
        for k, v in part_candidates.items():
            candidates[k] |= v

        del s1_slice, s2_slice, s3_slice, s1_part, s2_part, s3_part, part_candidates
        gc.collect()

    del s1_labels, s2_labels, s3_labels
    gc.collect()

    result = {}
    for s1_id in s1_df["entity_id"]:
        result[s1_id] = sorted(candidates.get(s1_id, set()))
    print("\nDone.")
    return result

In [11]:

import pandas as pd


def parse_id_list(cell):
    if pd.isna(cell) or str(cell).strip() == "":
        return set()
    return set(x.strip() for x in str(cell).split(",") if x.strip())


def load_ground_truth(path):
    """Returns dict: source1_entity_id -> set(matched_entity_ids)."""
    gt = pd.read_csv(path, sep="\t", dtype=str)
    return {row["source1_entity_id"]: parse_id_list(row["matched_entity_ids"])
            for _, row in gt.iterrows()}


def compute_blocking_recall(candidates: dict, ground_truth: dict) -> dict:
  
    total_true = 0
    total_found = 0
    per_entity_recall = []
    missed_by_entity = {}
    reduction_ratios = []

    for s1_id, true_matches in ground_truth.items():
        cand_set = set(candidates.get(s1_id, []))
        reduction_ratios.append(len(cand_set))

        if len(true_matches) == 0:
    
            continue

        found = true_matches & cand_set
        total_true += len(true_matches)
        total_found += len(found)
        entity_recall = len(found) / len(true_matches)
        per_entity_recall.append(entity_recall)
        if entity_recall < 1.0:
            missed_by_entity[s1_id] = true_matches - found

    macro_recall = sum(per_entity_recall) / len(per_entity_recall) if per_entity_recall else float("nan")
    micro_recall = total_found / total_true if total_true else float("nan")
    avg_candidates_per_entity = (sum(reduction_ratios) / len(reduction_ratios)
                                  if reduction_ratios else 0)

    return {
        "macro_recall": macro_recall,
        "micro_recall": micro_recall,
        "n_entities_with_matches": len(per_entity_recall),
        "n_entities_fully_missed": sum(1 for r in per_entity_recall if r == 0.0),
        "avg_candidates_per_entity": avg_candidates_per_entity,
        "missed_by_entity": missed_by_entity,
    }


def print_recall_report(stats: dict, top_n_examples: int = 10):
    print("=== Blocking Recall Report ===")
    print(f"Macro recall (per-entity avg): {stats['macro_recall']:.4f}")
    print(f"Micro recall (pooled):         {stats['micro_recall']:.4f}")
    print(f"Entities with >=1 true match:  {stats['n_entities_with_matches']}")
    print(f"Entities where ALL matches missed: {stats['n_entities_fully_missed']}")
    print(f"Avg candidates generated per S1 entity: {stats['avg_candidates_per_entity']:.1f}")
    if stats["missed_by_entity"]:
        print(f"\nExample entities with missed matches (showing up to {top_n_examples}):")
        for i, (s1_id, missed) in enumerate(stats["missed_by_entity"].items()):
            if i >= top_n_examples:
                break
            print(f"  {s1_id}: missed {sorted(missed)}")


## 5. Load Training Data

In [12]:
def read_tsv_with_progress(path, **read_csv_kwargs):
    """Read a large TSV in chunks, showing a progress bar based on approximate size."""
    file_size = os.path.getsize(path)
    chunk_size = 50_000  # rows per chunk -- lower this if RAM is tight

    chunks = []
    reader = pd.read_csv(path, sep="\t", dtype=str, chunksize=chunk_size, **read_csv_kwargs)

    if HAS_TQDM:
        pbar = tqdm(total=file_size, unit="B", unit_scale=True, desc=os.path.basename(path))
    for chunk in reader:
        chunks.append(chunk)
        if HAS_TQDM:
            chunk_bytes = chunk.memory_usage(deep=True).sum()
            pbar.update(min(chunk_bytes, file_size - pbar.n))
    if HAS_TQDM:
        pbar.close()

    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()
    return df.fillna("")

In [13]:
print("Reading train_source1.tsv...")
s1_train = read_tsv_with_progress(os.path.join(TRAIN_DIR, "train_source1.tsv"))

print("Reading train_source2.tsv...")
s2_train = read_tsv_with_progress(os.path.join(TRAIN_DIR, "train_source2.tsv"))

print("Reading train_source3.tsv...")
s3_train = read_tsv_with_progress(os.path.join(TRAIN_DIR, "train_source3.tsv"))

print("Reading train_ground_truth.tsv...")
ground_truth = load_ground_truth(os.path.join(TRAIN_DIR, "train_ground_truth.tsv"))

print(f"\nS1: {len(s1_train)} | S2: {len(s2_train)} | S3: {len(s3_train)} | GT entries: {len(ground_truth)}")
s1_train.head()

Reading train_source1.tsv...


train_source1.tsv: 100%|██████████| 210M/210M [00:08<00:00, 25.9MB/s] 


Reading train_source2.tsv...


train_source2.tsv: 100%|██████████| 489M/489M [00:19<00:00, 25.1MB/s] 


Reading train_source3.tsv...


train_source3.tsv: 100%|██████████| 504M/504M [00:20<00:00, 24.9MB/s] 


Reading train_ground_truth.tsv...

S1: 2206821 | S2: 5034616 | S3: 5285603 | GT entries: 2206821


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


## 6. Validation Split + Recall Check

Holds out a random subset of Source-1 entities. Source-2/Source-3 are **not** split — blocking always searches the full pool, mirroring test time.

In [14]:
import random

VAL_FRAC = 0.2
TOP_K = 10
SEED = 42

random.seed(SEED)
all_s1_ids = list(s1_train["entity_id"])
random.shuffle(all_s1_ids)
n_val = max(1, int(len(all_s1_ids) * VAL_FRAC))
val_ids = set(all_s1_ids[:n_val])

s1_val = s1_train[s1_train["entity_id"].isin(val_ids)].reset_index(drop=True)
print(f"Validation split: {len(s1_val)} / {len(s1_train)} S1 entities")

Validation split: 441364 / 2206821 S1 entities


In [ ]:

s1_val_500 = s1_val.iloc[:500].copy()

val_ids_500 = set(s1_val_500["entity_id"])

val_ground_truth_500 = {
    k: v
    for k, v in ground_truth.items()
    if k in val_ids_500
}
dc
candidates_500 = generate_all_candidates(
    s1_val_500,
    s2_train,
    s3_train,
    top_k=TOP_K
)

stats_500 = compute_blocking_recall(
    candidates_500,
    val_ground_truth_500
)

print_recall_report(stats_500)

Labeling countries (lightweight, full dataset)...
Found 2 country partitions.



Countries:   0%|          | 0/2 [00:01<?, ?country/s, india (S1:196 S2:2017799 S3:2115547)]

    S2: phonetic blocking...
    S2: sorted-token blocking...
    S2: TF-IDF blocking...
      Building TF-IDF index for 2,017,799 records...


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


      TF-IDF matrix: 2,017,799 x 10,000
      TF-IDF query: 196/196
    S3: phonetic blocking...
    S3: sorted-token blocking...
    S3: TF-IDF blocking...
      Building TF-IDF index for 2,115,547 records...


...

    Merging S2 + S3 candidates...


Countries:  50%|█████     | 1/2 [25:13<25:11, 1511.87s/country, united states (S1:304 S2:3016817 S3:3170056)]

    S2: phonetic blocking...
    S2: sorted-token blocking...
    S2: TF-IDF blocking...
      Building TF-IDF index for 3,016,817 records...


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


      TF-IDF matrix: 3,016,817 x 10,000


## 7. Generate `candidate_pairs.tsv` on the Test Set

Once validation recall is 95%+, run this against the real test data.

In [ ]:
s1_test = pd.read_csv(os.path.join(TEST_DIR, "test_source1.tsv"), sep="\t", dtype=str).fillna("")
s2_test = pd.read_csv(os.path.join(TEST_DIR, "test_source2.tsv"), sep="\t", dtype=str).fillna("")
s3_test = pd.read_csv(os.path.join(TEST_DIR, "test_source3.tsv"), sep="\t", dtype=str).fillna("")

print(f"Test S1: {len(s1_test)} | S2: {len(s2_test)} | S3: {len(s3_test)}")

test_candidates = generate_all_candidates(s1_test, s2_test, s3_test, top_k=TOP_K)

rows = [{"source1_entity_id": s1_id, "candidate_entity_ids": ",".join(cand_ids)}
        for s1_id, cand_ids in test_candidates.items()]
candidate_df = pd.DataFrame(rows)
out_path = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
candidate_df.to_csv(out_path, sep="\t", index=False)
print(f"Wrote {len(candidate_df)} rows to {out_path}")
candidate_df.head()

## Next step

`output/candidate_pairs.tsv` is the file your matching model runs inference over. Building and scoring that matching model (feature engineering on the pairs, a classifier, then narrowing candidates down into `matching_results.tsv`) is the next stage, separate from this blocking notebook.